In [1]:
import numpy as np

class CuckooSearchEV:
    def __init__(self, candidates, user_locations, n_nests=15, iterations=100, pa=0.25):
        self.candidates = candidates
        self.users = user_locations
        self.n_nests = n_nests
        self.iterations = iterations
        self.pa = pa
        self.n_stations = len(candidates)
        self.nests = np.random.randint(0, 2, size=(self.n_nests, self.n_stations))

    def fitness(self, nest):
        if np.sum(nest) == 0:
            return float('inf')

        active_stations = self.candidates[nest == 1]
        cost = np.sum(nest) * 5000

        distances = np.sqrt(((self.users[:, np.newaxis, :] - active_stations[np.newaxis, :, :]) ** 2).sum(axis=-1))
        min_distances = np.min(distances, axis=1)

        total_travel = np.sum(min_distances)
        coverage_penalty = np.sum(min_distances > 20) * 1000

        return cost + total_travel + coverage_penalty

    def levy_flight(self, nest):
        new_nest = nest.copy()
        step_size = int(np.round(np.abs(np.random.standard_cauchy())))
        if step_size > 0:
            idx = np.random.choice(self.n_stations, size=min(step_size, self.n_stations), replace=False)
            new_nest[idx] = 1 - new_nest[idx]
        return new_nest

    def solve(self):
        fitness_scores = np.array([self.fitness(n) for n in self.nests])
        best_idx = np.argmin(fitness_scores)
        best_nest = self.nests[best_idx].copy()
        best_fitness = fitness_scores[best_idx]

        for _ in range(self.iterations):
            for i in range(self.n_nests):
                new_nest = self.levy_flight(self.nests[i])
                new_fit = self.fitness(new_nest)

                rand_idx = np.random.randint(0, self.n_nests)
                if new_fit < fitness_scores[rand_idx]:
                    self.nests[rand_idx] = new_nest
                    fitness_scores[rand_idx] = new_fit

            worst_indices = np.argsort(fitness_scores)[-int(self.n_nests * self.pa):]
            for idx in worst_indices:
                self.nests[idx] = np.random.randint(0, 2, size=self.n_stations)
                fitness_scores[idx] = self.fitness(self.nests[idx])

            current_best_idx = np.argmin(fitness_scores)
            if fitness_scores[current_best_idx] < best_fitness:
                best_fitness = fitness_scores[current_best_idx]
                best_nest = self.nests[current_best_idx].copy()

        return best_nest, best_fitness

if __name__ == "__main__":
    np.random.seed(42)
    candidate_spots = np.random.rand(10, 2) * 100
    user_spots = np.random.rand(50, 2) * 100

    cs = CuckooSearchEV(candidate_spots, user_spots)
    best_config, min_cost = cs.solve()

    print("Best Configuration Vector:", best_config)
    print("Total Optimized Score Value:", round(min_cost, 2))


Best Configuration Vector: [0 1 0 1 0 0 1 1 1 0]
Total Optimized Score Value: 41833.18
